## Step 10 — revised zones
**# of cells in notebook:** 2

**Purpose:** Dissolve blocks into singlepart groupings of shared initial zone--density cluster--admin values. Calculate their populations. 

**Input:**

- a geodatabase with the blocks layer from previous steps
  
**Output:** `zones_2` layer with new fields `concat`, `zone2_id`, `population`, `block_count`

**Main logic:**

1. Create `concat` field on the original blocks layer by combining `initial_zones_1_sj`, `geoboundaries`, and `cluster_revised`. This combined value defines which blocks belong to the same revised zone group.
2. Dissolve the blocks by `concat` to create `zones_2`, using `SINGLE_PART` so disconnected polygons with the same `concat` value remain as separate features. Population is intentionally not summed during the dissolve because dissolve statistics would summarize by concat group rather than by each singlepart zone feature.
3. Add a stable `zone2_id` field to `zones_2`, using each zone feature’s `OBJECTID` as its identifier. This creates a reliable ID for assigning block populations back to the correct singlepart zone.
4. Convert the original blocks to inside-constrained points, then spatially join those block points to `zones_2` so each block receives the `zone2_id` of the singlepart zone that contains it.
5. Sum block population by `zone2_id`, write the resulting population and `block_count` values back to `zones_`2, and run QA checks to confirm population totals are conserved and all block points were assigned to a zone.
6. Second cell is purely diagnostic QA/QC to identify `initial_zones_1_sj` with populations <100 for review/inspection.

In [ ]:
import arcpy
import os
from collections import defaultdict

# ============================================================
# USER SETTINGS
# ============================================================

blocks_fc = r"E:\World Bank deliverbale 1\_analysis\blocks\blocks.gdb\juba_blocks_20260415_small_utm36n"

out_gdb = r"E:\World Bank deliverbale 1\_analysis\zones\zones.gdb"
out_fc_name = "zones_2"
out_fc = os.path.join(out_gdb, out_fc_name)

# Fields used to define zones
group_fields = [
    "initial_zones_1_sj",
    "geoboundaries",
    "cluster_revised"
]

# Population field in the blocks layer
population_field = "population"

# Field to create/update on blocks
concat_field = "concat"

# Stable ID field to create on zones_2
zone2_id_field = "zone2_id"

# Temporary / inspection outputs
block_points_fc = os.path.join(out_gdb, "zones_2_block_points_inside")
block_points_join_fc = os.path.join(out_gdb, "zones_2_block_points_joined")
zone_population_table = os.path.join(out_gdb, "zones_2_population_summary")

# Concatenation settings
NULL_TEXT = "NULL"
CONCAT_DELIMITER = "_"

OVERWRITE_OUTPUT = True

# ============================================================
# ENVIRONMENT
# ============================================================

arcpy.env.overwriteOutput = OVERWRITE_OUTPUT

print("============================================================")
print("Create singlepart zones_2 with correct per-feature population")
print("============================================================")
print(f"Input blocks: {blocks_fc}")
print(f"Output zones: {out_fc}")
print("")
print("Grouping fields:")
for f in group_fields:
    print(f"  {f}")
print(f"Population field: {population_field}")
print(f"Concat field:     {concat_field}")
print(f"Zone ID field:    {zone2_id_field}")

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def clean_value_for_concat(value):
    if value is None:
        return NULL_TEXT

    value_text = str(value).strip()

    if value_text == "" or value_text.lower() == "<null>":
        return NULL_TEXT

    value_text = value_text.replace(CONCAT_DELIMITER, "-")

    return value_text


def delete_if_exists(path):
    if arcpy.Exists(path):
        print(f"Deleting existing output: {path}")
        arcpy.management.Delete(path)


def sum_field(table, field_name):
    total = 0.0
    nonnull_count = 0
    null_count = 0

    with arcpy.da.SearchCursor(table, [field_name]) as cursor:
        for (value,) in cursor:
            if value is None:
                null_count += 1
                continue

            total += float(value)
            nonnull_count += 1

    return total, nonnull_count, null_count


# ============================================================
# VALIDATE INPUTS
# ============================================================

if not arcpy.Exists(blocks_fc):
    raise FileNotFoundError(f"Blocks layer does not exist: {blocks_fc}")

if not arcpy.Exists(out_gdb):
    raise FileNotFoundError(f"Output geodatabase does not exist: {out_gdb}")

existing_fields = {f.name for f in arcpy.ListFields(blocks_fc)}

required_fields = set(group_fields + [population_field])
missing_fields = required_fields - existing_fields

if missing_fields:
    raise ValueError(f"Missing required fields in blocks layer: {missing_fields}")

blocks_oid_field = arcpy.Describe(blocks_fc).OIDFieldName

input_count = int(arcpy.management.GetCount(blocks_fc)[0])
input_pop_sum, input_pop_nonnull_count, input_pop_null_count = sum_field(blocks_fc, population_field)

print("")
print("Input checks:")
print(f"  Block count:                  {input_count:,}")
print(f"  Input population sum:         {input_pop_sum:,.6f}")
print(f"  Non-null population records:  {input_pop_nonnull_count:,}")
print(f"  Null population records:      {input_pop_null_count:,}")

# ============================================================
# CLEAN OLD OUTPUTS
# ============================================================

for item in [
    out_fc,
    block_points_fc,
    block_points_join_fc,
    zone_population_table
]:
    delete_if_exists(item)

# ============================================================
# STEP 1: CREATE / UPDATE concat FIELD ON BLOCKS
# ============================================================

print("\n============================================================")
print("Step 1: Creating/updating concat field on blocks")
print("============================================================")

existing_fields = {f.name for f in arcpy.ListFields(blocks_fc)}

if concat_field not in existing_fields:
    print(f"Adding field: {concat_field}")
    arcpy.management.AddField(
        in_table=blocks_fc,
        field_name=concat_field,
        field_type="TEXT",
        field_length=255
    )
else:
    print(f"Field already exists; values will be overwritten: {concat_field}")

update_fields = group_fields + [concat_field]

unique_concat_values = set()
updated = 0

with arcpy.da.UpdateCursor(blocks_fc, update_fields) as cursor:
    for row in cursor:

        source_values = row[:-1]

        cleaned_values = [
            clean_value_for_concat(value)
            for value in source_values
        ]

        concat_value = CONCAT_DELIMITER.join(cleaned_values)

        row[-1] = concat_value
        cursor.updateRow(row)

        unique_concat_values.add(concat_value)
        updated += 1

print(f"Rows updated:          {updated:,}")
print(f"Unique concat values:  {len(unique_concat_values):,}")

# ============================================================
# STEP 2: DISSOLVE BY concat AS SINGLEPART
#
# Important:
# We do NOT sum population here, because Dissolve statistics are
# calculated at the concat-group level, not the singlepart-feature level.
# ============================================================

print("\n============================================================")
print("Step 2: Dissolving by concat as SINGLE_PART")
print("============================================================")

print("Running Dissolve...")
print(f"  Dissolve field:     {concat_field}")
print("  Multipart setting:  SINGLE_PART")
print("  Population stats:   skipped here; calculated later per singlepart zone")

arcpy.management.Dissolve(
    in_features=blocks_fc,
    out_feature_class=out_fc,
    dissolve_field=[concat_field],
    statistics_fields=[],
    multi_part="SINGLE_PART",
    unsplit_lines="DISSOLVE_LINES"
)

print(f"Created zones_2: {out_fc}")

zone_count = int(arcpy.management.GetCount(out_fc)[0])
print(f"zones_2 feature count: {zone_count:,}")

# ============================================================
# STEP 3: ADD STABLE zone2_id TO zones_2
# ============================================================

print("\n============================================================")
print("Step 3: Adding stable zone2_id field")
print("============================================================")

zones_oid_field = arcpy.Describe(out_fc).OIDFieldName

existing_zone_fields = {f.name for f in arcpy.ListFields(out_fc)}

if zone2_id_field not in existing_zone_fields:
    arcpy.management.AddField(out_fc, zone2_id_field, "LONG")
    print(f"Added field: {zone2_id_field}")
else:
    print(f"Field already exists; values will be overwritten: {zone2_id_field}")

with arcpy.da.UpdateCursor(out_fc, [zones_oid_field, zone2_id_field]) as cursor:
    for oid, zone2_id in cursor:
        cursor.updateRow([oid, oid])

print(f"Calculated {zone2_id_field} from {zones_oid_field}")

# ============================================================
# STEP 4: CREATE INSIDE POINTS FROM ORIGINAL BLOCKS
#
# These points carry each block's attributes, including population.
# INSIDE ensures the point falls inside its source block polygon.
# ============================================================

print("\n============================================================")
print("Step 4: Creating inside block points")
print("============================================================")

arcpy.management.FeatureToPoint(
    in_features=blocks_fc,
    out_feature_class=block_points_fc,
    point_location="INSIDE"
)

print(f"Created block inside points: {block_points_fc}")

point_count = int(arcpy.management.GetCount(block_points_fc)[0])
print(f"Block point count: {point_count:,}")

if point_count != input_count:
    print("WARNING: block point count does not match input block count.")

# ============================================================
# STEP 5: SPATIALLY JOIN BLOCK POINTS TO zones_2
#
# Each block point receives the zone2_id of the singlepart zone
# that contains it.
# ============================================================

print("\n============================================================")
print("Step 5: Spatially joining block points to zones_2")
print("============================================================")

arcpy.analysis.SpatialJoin(
    target_features=block_points_fc,
    join_features=out_fc,
    out_feature_class=block_points_join_fc,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    match_option="WITHIN"
)

print(f"Created joined block points: {block_points_join_fc}")

joined_count = int(arcpy.management.GetCount(block_points_join_fc)[0])
print(f"Joined point count: {joined_count:,}")

# Check expected fields
joined_fields = {f.name for f in arcpy.ListFields(block_points_join_fc)}

if zone2_id_field not in joined_fields:
    raise ValueError(
        f"Spatial join output does not contain {zone2_id_field}. "
        "Inspect fields in the joined point output."
    )

if population_field not in joined_fields:
    raise ValueError(
        f"Spatial join output does not contain {population_field}. "
        "Inspect fields in the joined point output."
    )

# ============================================================
# STEP 6: SUM BLOCK POPULATION BY zone2_id
# ============================================================

print("\n============================================================")
print("Step 6: Summing block population by zone2_id")
print("============================================================")

pop_by_zone2_id = defaultdict(float)
block_count_by_zone2_id = defaultdict(int)

unmatched_points = 0
null_population_points = 0

with arcpy.da.SearchCursor(
    block_points_join_fc,
    [zone2_id_field, population_field]
) as cursor:

    for zone2_id, pop_value in cursor:

        if zone2_id is None or zone2_id == -1:
            unmatched_points += 1
            continue

        block_count_by_zone2_id[zone2_id] += 1

        if pop_value is None:
            null_population_points += 1
            continue

        pop_by_zone2_id[zone2_id] += float(pop_value)

print(f"Zones with assigned block population: {len(pop_by_zone2_id):,}")
print(f"Unmatched block points:               {unmatched_points:,}")
print(f"Null population block points:          {null_population_points:,}")

# ============================================================
# STEP 7: WRITE OPTIONAL POPULATION SUMMARY TABLE
# ============================================================

print("\n============================================================")
print("Step 7: Writing population summary table")
print("============================================================")

delete_if_exists(zone_population_table)

arcpy.management.CreateTable(out_gdb, os.path.basename(zone_population_table))

arcpy.management.AddField(zone_population_table, zone2_id_field, "LONG")
arcpy.management.AddField(zone_population_table, "population", "DOUBLE")
arcpy.management.AddField(zone_population_table, "block_count", "LONG")

with arcpy.da.InsertCursor(
    zone_population_table,
    [zone2_id_field, "population", "block_count"]
) as cursor:

    for zone2_id in sorted(block_count_by_zone2_id.keys()):
        cursor.insertRow([
            zone2_id,
            pop_by_zone2_id.get(zone2_id, 0.0),
            block_count_by_zone2_id.get(zone2_id, 0)
        ])

print(f"Created population summary table: {zone_population_table}")

# ============================================================
# STEP 8: ADD / UPDATE population AND block_count ON zones_2
# ============================================================

print("\n============================================================")
print("Step 8: Writing population back to zones_2")
print("============================================================")

zone_fields = {f.name for f in arcpy.ListFields(out_fc)}

if population_field not in zone_fields:
    arcpy.management.AddField(out_fc, population_field, "DOUBLE")
    print(f"Added field to zones_2: {population_field}")
else:
    print(f"Field already exists; values will be overwritten: {population_field}")

block_count_field = "block_count"

if block_count_field not in zone_fields:
    arcpy.management.AddField(out_fc, block_count_field, "LONG")
    print(f"Added field to zones_2: {block_count_field}")
else:
    print(f"Field already exists; values will be overwritten: {block_count_field}")

updated_zones = 0
zones_without_blocks = 0

with arcpy.da.UpdateCursor(
    out_fc,
    [zone2_id_field, population_field, block_count_field]
) as cursor:

    for zone2_id, old_pop, old_block_count in cursor:

        new_pop = pop_by_zone2_id.get(zone2_id, 0.0)
        new_block_count = block_count_by_zone2_id.get(zone2_id, 0)

        if new_block_count == 0:
            zones_without_blocks += 1

        cursor.updateRow([zone2_id, new_pop, new_block_count])
        updated_zones += 1

print(f"Updated zones:        {updated_zones:,}")
print(f"Zones without blocks: {zones_without_blocks:,}")

# ============================================================
# STEP 9: QA CHECKS
# ============================================================

print("\n============================================================")
print("Step 9: QA checks")
print("============================================================")

output_pop_sum, output_pop_nonnull_count, output_pop_null_count = sum_field(out_fc, population_field)

print("Population conservation check:")
print(f"  Input block population sum:   {input_pop_sum:,.6f}")
print(f"  zones_2 population sum:       {output_pop_sum:,.6f}")
print(f"  Difference output - input:    {output_pop_sum - input_pop_sum:,.6f}")

print("")
print("Feature count check:")
print(f"  Unique concat values:         {len(unique_concat_values):,}")
print(f"  zones_2 singlepart features:  {zone_count:,}")

if zone_count >= len(unique_concat_values):
    print("  OK: singlepart output can have more features than unique concat groups.")
else:
    print("  WARNING: zones_2 has fewer features than unique concat groups.")

print("")
print("Point assignment check:")
print(f"  Input blocks:                 {input_count:,}")
print(f"  Inside block points:          {point_count:,}")
print(f"  Joined block points:          {joined_count:,}")
print(f"  Unmatched block points:       {unmatched_points:,}")

if unmatched_points == 0:
    print("  OK: every block point was assigned to a zones_2 feature.")
else:
    print("  WARNING: some block points were not assigned to zones_2.")

print("")
print("Outputs:")
print(f"  zones_2:                      {out_fc}")
print(f"  block inside points:          {block_points_fc}")
print(f"  joined block points:          {block_points_join_fc}")
print(f"  population summary table:     {zone_population_table}")

print("\nDone.")

In [ ]:
import arcpy
from collections import defaultdict

# ------------------------------------------------------------
# Input
# ------------------------------------------------------------

blocks_fc = r"E:\World Bank deliverbale 1\_analysis\blocks\blocks.gdb\juba_blocks_20260415_small_utm36n"

zone_field = "initial_zones_1_sj"
population_field = "population"

pop_threshold = 100

# ------------------------------------------------------------
# Validate fields
# ------------------------------------------------------------

fields = [f.name for f in arcpy.ListFields(blocks_fc)]
field_set = set(fields)

if zone_field not in field_set:
    similar = [
        f for f in fields
        if "initial" in f.lower() or "zone" in f.lower()
    ]

    print(f"ERROR: Field not found: {zone_field}")
    print("\nSimilar zone-related fields found:")
    for f in similar:
        print(f"  {f}")

    raise ValueError(f"Field not found: {zone_field}")

if population_field not in field_set:
    raise ValueError(f"Field not found: {population_field}")

# ------------------------------------------------------------
# Sum population by initial zone
# ------------------------------------------------------------

pop_by_zone = defaultdict(float)
count_by_zone = defaultdict(int)
null_pop_count_by_zone = defaultdict(int)

with arcpy.da.SearchCursor(blocks_fc, [zone_field, population_field]) as cursor:
    for zone_value, pop_value in cursor:

        # Keep null initial zone values visible as a category
        if zone_value is None:
            zone_key = "<Null>"
        else:
            zone_key = zone_value

        count_by_zone[zone_key] += 1

        if pop_value is None:
            null_pop_count_by_zone[zone_key] += 1
            continue

        pop_by_zone[zone_key] += float(pop_value)

# ------------------------------------------------------------
# Print all zones sorted by population
# ------------------------------------------------------------

rows = []

for zone_key in count_by_zone.keys():
    rows.append({
        "initial_zone": zone_key,
        "population_sum": pop_by_zone.get(zone_key, 0.0),
        "block_count": count_by_zone.get(zone_key, 0),
        "null_population_count": null_pop_count_by_zone.get(zone_key, 0)
    })

rows_sorted = sorted(rows, key=lambda r: r["population_sum"])

print("============================================================")
print(f"Population by {zone_field}")
print("============================================================")
print(f"{'initial_zone':>15}  {'population_sum':>18}  {'blocks':>8}  {'null_pop':>8}")
print("-" * 60)

for r in rows_sorted:
    print(
        f"{str(r['initial_zone']):>15}  "
        f"{r['population_sum']:18,.3f}  "
        f"{r['block_count']:8,}  "
        f"{r['null_population_count']:8,}"
    )

# ------------------------------------------------------------
# Print only zones below threshold
# ------------------------------------------------------------

below_threshold = [
    r for r in rows_sorted
    if r["population_sum"] < pop_threshold
]

print("\n============================================================")
print(f"Initial zones with population < {pop_threshold}")
print("============================================================")
print(f"Count: {len(below_threshold):,}")
print("")
print(f"{'initial_zone':>15}  {'population_sum':>18}  {'blocks':>8}  {'null_pop':>8}")
print("-" * 60)

for r in below_threshold:
    print(
        f"{str(r['initial_zone']):>15}  "
        f"{r['population_sum']:18,.3f}  "
        f"{r['block_count']:8,}  "
        f"{r['null_population_count']:8,}"
    )

# ------------------------------------------------------------
# Overall check
# ------------------------------------------------------------

total_pop = sum(r["population_sum"] for r in rows)
total_blocks = sum(r["block_count"] for r in rows)

print("\n============================================================")
print("Overall check")
print("============================================================")
print(f"Number of unique initial zones: {len(rows):,}")
print(f"Total blocks:                   {total_blocks:,}")
print(f"Total population:               {total_pop:,.3f}")
print("Done.")